# 1.3 标准 Attention 的访存瓶颈

上一节我们搞清楚了 Attention **在算什么**。本节回答：放到真实加速器上，这套三步计算**为什么慢**。

结论先行（学完本节你将能亲手算出这些数字）：

- 标准实现拆成三个 Kernel：两个矩阵乘 + 一个 Softmax；
- **Softmax 是访存受限（memory-bound）算子**，几乎不吃算力、纯搬数据；
- 更糟的是 `S`、`P` 两个 `n × n` 的中间矩阵要**写回显存再读回来**，与有效数据相比是纯粹的额外开销；
- 序列越长（n 越大），这笔开销按 `n²` 增长——比有效计算（`n²·d`）中不含 d 的部分恶化得更快。

## 一、加速器的存储层次：算得快，但拿数据慢

![存储层次](images/storage_hierarchy.svg)

任何现代加速器（NPU / GPU）的计算单元都极快，真正的鸿沟在**数据搬运**。存储层次是金字塔结构：

| 层级 | 典型容量 | 典型带宽（量级） | 角色 |
|--|--|--|--|
片上 SRAM / Unified Buffer | 数百 KB | 100+ TB/s | Kernel 内部的工作区，**放不下大矩阵** |
L2 Cache | 数十 MB | ~10 TB/s | 多个 AI Core 共享 |
HBM 显存 | 数十 GB | ~1-3 TB/s | Q、K、V、O 的家，一切输入输出的必经之路 |

注意两点：

1. **带宽差一到两个数量级**：数据每从 HBM 搬一次，都付 10~100 倍的时间代价；
2. **片上容量极小**：256 KB 量级连一个 `2048×2048` 的 fp16 矩阵（8 MB）都放不下。

于是所有算子优化的核心问题可以一句话概括：**如何让数据搬上金字塔顶端一次，就被复用到底？**

## 二、算术强度：判断算子是算力瓶颈还是带宽瓶颈

### 算术强度（Arithmetic Intensity）

$$\text{AI} = \frac{\text{计算量 (FLOPs)}}{\text{访存量 (Bytes, 从 HBM)}}$$

它回答：每从显存搬 1 字节，能换来多少次浮点运算。

### 屋顶线模型（Roofline）

一个 Kernel 的性能上限是两条“屋顶”的较小者：

$$\text{性能} \le \min\left(\; \text{峰值算力} \;，\; \text{AI} \times \text{峰值带宽} \;\right)$$

- **AI 大**（如大矩阵乘，AI 可达数百）：算力先到顶 → **compute-bound（算力受限）**；
- **AI 小**（如逐元素操作，AI ≈ 1~2）：带宽先到顶 → **memory-bound（访存受限）**，计算单元大量空转。

两个例子建立手感：

- `4096×4096` 的 GEMM：每读入一对小分块完成大量乘加，数据被复用上千次 → AI 很高；
- 逐元素 Softmax（行内归一化）：每个元素读进来只做一两次 exp/除法 → **AI ≈ 1**，典型 memory-bound。

In [ ]:
import numpy as np

def flops_gemm(M, N, K):
    """C = A·B 的 FLOPs：每个输出元素 K 次乘 + K 次加。"""
    return 2 * M * N * K

def bytes_gemm(M, N, K, dt=2):
    """从 HBM 读 A、B，写 C 的字节数（dt=2 表示 fp16）。"""
    return (M*K + K*N + M*N) * dt

M = N = K = 4096
print(f'GEMM {M}x{N}x{K}:')
print(f'  FLOPs = {flops_gemm(M,N,K)/1e12:.1f} TFLOPs')
print(f'  Bytes = {bytes_gemm(M,N,K)/1e9:.2f} GB')
print(f'  算术强度 AI = {flops_gemm(M,N,K)/bytes_gemm(M,N,K):.1f} FLOPs/Byte  →  compute-bound')

## 三、标准实现的数据流：中间矩阵在 HBM 反复往返

标准实现（也是各类框架 naive 路径的做法）把三步拆成三个独立 Kernel，各自从 HBM 读输入、把输出写回 HBM：

![标准实现数据流](images/standard_attention_dataflow.svg)

每个 Kernel 单独看都还合理，但串起来看：

1. Kernel 1 算完 `S`（n×n）→ **写回 HBM**；
2. Kernel 2 从 HBM **读回 S**（求行最大值、行和各扫一遍，共两趟），算出 `P`（n×n）→ 又**写回 HBM**；
3. Kernel 3 从 HBM **读回 P**，乘 V 得 O。

`S` 和 `P` 都是没人最终需要的**中间结果**：`S` 写 1 次、读 2 次，`P` 写 1 次、读 1 次——合计 **5 次穿过最慢的 HBM 通道，fp16 下约 10n² 字节的额外流量**（即下文 `standard_attention_bytes` 代码所建的模型；若 Softmax 还需回写中间量，实际只会更多）。

请记住这张图右下角那句加粗的话——**这就是 FlashAttention 要消灭的东西**。

## 四、定量分析：亲手算出标准实现的账单

约定：fp16（每元素 2 字节），`n` 个 token，头维 `d`，单头单次 Attention。

In [ ]:
dt = 2  # fp16 每元素字节数

def standard_attention_flops(n, d):
    """三步计算的 FLOPs（Softmax 的 exp/求和按 5n^2 估算）。"""
    qk   = 2 * n * n * d          # S = Q·K^T
    smx  = 5 * n * n              # 行最大、行和、exp、除
    pv   = 2 * n * n * d          # O = P·V
    return qk + smx + pv

def standard_attention_bytes(n, d):
    """三 Kernel 各自独立时，穿越 HBM 的总字节数。"""
    qk  = (n*d + n*d) * dt + n*n * dt          # 读 Q,K；写 S
    smx = (n*n) * dt * 2 + n*n * dt            # 读 S(最大+求和两遍)、写 P（保守估计）
    pv  = (n*n + n*d) * dt + n*d * dt          # 读 P,V；写 O
    return qk + smx + pv

def flash_attention_bytes(n, d, M):
    """FlashAttention 的 HBM 流量：Q/K/V/O 各一次，外加 KV 因分块被重读约 n/M 次。"""
    qkv_o = 4 * n * d * dt
    kv_reread = 2 * n * d * dt * (n / M)       # K、V 各被 n/M 个 Q 块复用
    return qkv_o + kv_reread

for n in [1024, 4096, 16384]:
    d = 128
    std_B  = standard_attention_bytes(n, d)
    fl_B   = flash_attention_bytes(n, d, M=128)
    print(f'n={n:6d}: 标准访存 {std_B/1e9:8.2f} GB | Flash 访存 {fl_B/1e9:6.2f} GB |'
          f' 比值 {std_B/fl_B:5.1f}x')

可以看到两个事实：

1. 标准实现比 FlashAttention 多出的访存，主要就是中间矩阵那笔账——`10n²` 对 `4n²` 的流量差，S、P 的 HBM 往返被消灭了；
2. 但 FlashAttention 也**不是“线性访存”**：KV 重读项 `2·n·d·dt·(n/M)` 本身仍是 `n²` 量级（本例 `d=M=128` 时恰为 `4n²`，系数不足标准实现的一半），所以两列数字仍同阶增长，比值稳定在 2.2→2.5 倍——正是上面打印出的结果。

想把重读项也压下去，靠两个杠杆：**增大分块 M**（重读次数按 `1/M` 下降，前提是片上放得下）；让重读尽量命中 **L2 Cache**（几十 MB、能缓存不少 KV 块），不必每次都下到 HBM——真实工程实现（FA-2/FA-3）正是沿这两个方向继续优化。

### 中间矩阵到底有多大？

再看一个直观的数字——`S` 矩阵自己的体积：

In [ ]:
d = 128
for n in [1024, 4096, 16384, 65536]:
    S_bytes = n * n * dt
    print(f'n = {n:6d} 时：单个 S 矩阵 = {S_bytes/1e6:8.1f} MB（片上 SRAM 只有 ~0.25 MB，差 {S_bytes/0.25e6:6.0f} 倍）')

## 五、Roofline 账单：谁在拖慢整体？

最后把三笔账合起来，假设一个典型加速器：**峰值算力 300 TFLOPS（fp16）、峰值 HBM 带宽 1.6 TB/s**。逐个 Kernel 估算运行时间（取两个“屋顶”的较小者）：

In [ ]:
PEAK_FLOPS = 300e12   # 300 TFLOPS
PEAK_BW    = 1.6e12   # 1.6 TB/s

def kernel_time(name, flops, bytes_):
    ai = flops / max(bytes_, 1)
    t_compute = flops / PEAK_FLOPS
    t_memory  = bytes_ / PEAK_BW
    bound = '算力受限' if t_compute > t_memory else '访存受限'
    print(f'{name:12s} AI = {ai:7.1f} FLOP/Byte | 纯算 {t_compute*1e3:7.3f} ms |'
          f' 纯访存 {t_memory*1e3:7.3f} ms | 实际 ≈ {max(t_compute, t_memory)*1e3:7.3f} ms ({bound})')
    return max(t_compute, t_memory)

n, d = 4096, 128
print(f'场景：n = {n}, d = {d}, fp16, 单头\n')

t1 = kernel_time('① QK^T',   2*n*n*d,        (n*d + n*d + n*n)*dt)
t2 = kernel_time('② Softmax', 5*n*n,          n*n*dt*3)          # 读 S 两遍 + 写 P
t3 = kernel_time('③ P·V',     2*n*n*d,        (n*n + n*d + n*d)*dt)
print(f'\n三个 Kernel 合计 ≈ {(t1+t2+t3)*1e3:.2f} ms，其中访存受限的 Softmax 就占了 {t2/(t1+t2+t3)*100:.0f}%')

### 结论

1. **Softmax Kernel 是纯粹的访存受限**：AI ≈ 1~2，每搬 1 字节只换 1 次左右的运算——计算单元几乎全程空转，一个“最闲”的 Kernel 却拿走了最多时间；
2. **两个 GEMM 也被拖下了水**：纯矩阵乘的 AI 可达数百（本应打满算力），但写 S / 读 P 的 `n²` 量级流量把它们的 AI 压到 `≈ d`（此处 128），跌破屋顶线拐点，同样沦为访存受限——中间矩阵的往返把整个算子变成了“搬运工”；
3. 额外流量 `≈ 10n²` 字节（fp16，即 5 次穿越 × n² 个元素 × 2 字节）随序列长度平方增长，**n=4096 时约 170 MB，n=16384 时约 2.7 GB**（单头！乘上头数和 batch 还要翻几十倍）。

### 病根与药方

病根不在计算，而在**数据流**：中间结果不该回显存。药方也就呼之欲出——

> **把三个 Kernel 融合成一个（Fused），把 Q、K、V 切成小块，让 S、P 只存在于片上，算完整行才把 O 写出去。**

这就是 FlashAttention 的出发点。但要这么做，必须先解决两个拦路虎：

- 片上一次只有**一个块**，而 Softmax 归一化需要**整行**的信息（行最大值、行和）——怎么办？→ **Online Softmax**（1.4 节）；
- 块算完后怎么保证结果和整体计算**精确一致**？→ 三步递推的数学等价性（1.4、1.5 节验证）。

## 章节测验

1. 写出算术强度的定义。AI = 100 和 AI = 1 的算子分别受什么限制？
2. 标准实现中，`S` 和 `P` 两个矩阵合计穿越 HBM 几次？大约多少额外字节（用 n 表示）？
3. `n = 4096, d = 128, fp16` 时，单个 S 矩阵多大？和 256KB 的片上 SRAM 差多少倍？
4. 为什么说“Softmax 是访存受限算子”？用 AI 的数量级说明。
5. （思考题）如果把三个 Kernel 融合成一个、中间结果不落 HBM，会立刻遇到什么数学障碍？（提示：归一化需要什么信息？）

> 答案见 `answer/01.03_answer.txt`。

下一节：[1.4 FlashAttention 核心原理](01.04_fa_principle.ipynb)——两只会拦路的“老虎”如何被 Online Softmax 和分块计算驯服。